# 00 ・ 環境設定

跑這一本，確認你的環境可以上課，並且把整套系統的**地基**蓋好。
全部做完大約五分鐘。

## 這套教材在做什麼

把兩家影城「現在上映什麼」抓下來，補上電影資料庫的評分、海報、類型，
整合成一份清單，再讓 AI 根據這份清單回答問題，最後接成一個真的網頁。

| Notebook | 主題 | 產出 |
|---|---|---|
| `00_環境設定` | 地基：設定、金鑰、統一的網路出入口 | `movieapp/config.py`、`movieapp/http.py` |
| `01_影城API` | 沒有文件的 API、遞迴挖資料 | `movieapp/sources.py` |
| `02_TMDB` | 正規 API、金鑰、比對、平行查詢 | `movieapp/tmdb.py` |
| `03_資料整合` | 跨來源去重、pandas 篩選 | `movieapp/merge.py` |
| `04_Gemini對話` | 把資料交給 AI 回答 | `movieapp/gemini.py` |
| `05_接成服務` | 包成 API、寫出前端、跑起真的網頁 | `server/` 全部 + 完整系統 |

**請照順序做。** 每一本都會用 `%%writefile` 產生真正的檔案，
後面的章節和最後的網頁服務都靠這些檔案。

> **這包教材裡除了 notebook 沒有別的程式碼。**
> `movieapp/`、`server/` 兩個資料夾都是你在課程中一格一格寫出來的 ——
> 包含最後那個網頁。整包刪掉只留 `notebooks/`，照順序重跑就會長回來。

---
## 1 ・ 安裝套件

**用 `%pip` 而不是 `!pip`。** `%pip` 會裝到目前 kernel 的環境；
`!pip` 是丟給系統的命令列。用 Anaconda 時這兩者會裝到不同地方，
然後你會遇到「明明裝好了卻 import 不到」的經典狀況。

In [ ]:
%pip install -q requests pandas "Django>=4.2,<5.1" django-cors-headers

---
## 2 ・ 先蓋地基

後面四章你會寫出四個模組，但它們都要站在兩個東西上面：

| 檔案 | 負責什麼 |
|---|---|
| `movieapp/config.py` | 金鑰怎麼拿、環境檢查、中英混排的表格排版 |
| `movieapp/http.py` | **所有**對外的 HTTP 請求都走這裡 |

`http.py` 那句「所有」是這套設計的重點。外部 API 會逾時、會斷線、
會回 429、會回一坨不是 JSON 的東西 —— 如果每個地方各自處理，
你會寫四份幾乎一樣但細節都不同的錯誤處理。

收攏成一個出入口之後，這些只要寫一次；
之後想加快取、加重試、加日誌，也都只有那一個地方要改。

先把資料夾建出來 —— `%%writefile` 不會幫你建目錄，
目錄不存在它會直接報錯。

In [ ]:
from pathlib import Path

PKG = Path("../movieapp")
PKG.mkdir(parents=True, exist_ok=True)
print("建立完成：", PKG.resolve())

把 `movieapp` 變成一個真正的 Python 套件 —— 有 `__init__.py` 才 import 得到。

In [ ]:
%%writefile ../movieapp/__init__.py
"""電影整合系統的核心邏輯層。

這一層是「唯一真相」：notebook 和 Django 服務都只是它的使用者，
自己不藏任何邏輯。

裡面每一個 .py 都是 notebook 用 %%writefile 產生的：
  * config.py / http.py                      —— 00_環境設定
  * sources.py / tmdb.py / merge.py / gemini.py —— 01 ~ 04

所以這個資料夾可以整個刪掉，照順序重跑 notebook 就會長回來。
"""

__all__ = ["config", "http"]

### `config.py`：金鑰與環境

這一份你不用背，但值得看兩個地方：

1. **`get_key()` 的順序是「環境變數 → `.env` → 當場輸入」。**
   所以沒有 `.env` 也能上課，程式會在需要金鑰的那一刻跳出輸入框，
   而且輸入的內容不會留在 notebook 的輸出裡。
2. **`pad()` 為什麼要自己寫。** 中文字在畫面上佔兩格，但 `len()` 算一個字元，
   直接用 `ljust()` 排中英混雜的表格一定會歪。後面每一章印表格都會用到它。

In [ ]:
%%writefile ../movieapp/config.py
"""執行環境設定：金鑰載入、環境自我檢查、表格排版。

這個檔案是整包教材的地基，00~05 每一本 notebook 開頭都會呼叫 setup()。

本檔案由 notebooks/00_環境設定.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

from __future__ import annotations

import importlib.util
import os
import sys
import unicodedata
from pathlib import Path

# 專案根目錄（jupyter_MVP/）
PKG_DIR = Path(__file__).resolve().parent
ROOT = PKG_DIR.parent

# 金鑰的環境變數名稱
TMDB_KEY_NAME = "TMDB_READ_ACCESS_TOKEN"
GEMINI_KEY_NAME = "GEMINI_API_KEY"

# 哪個模組是哪一本 notebook 產生的（用來給出有用的錯誤訊息）
_MODULE_OWNER = {
    "sources": "01_影城API.ipynb",
    "tmdb": "02_TMDB.ipynb",
    "merge": "03_資料整合.ipynb",
    "gemini": "04_Gemini對話.ipynb",
}


# --------------------------------------------------------------------------
# 金鑰
# --------------------------------------------------------------------------
def env_path() -> Path:
    """.env 的位置。整個專案只有這裡決定它在哪。"""
    return ROOT / ".env"


def load_env(path: str | Path | None = None, override: bool = False) -> dict:
    """讀取 .env 檔並寫進 os.environ。

    極簡實作，不依賴 python-dotenv —— 只認 KEY=VALUE，# 開頭是註解。
    .env 不存在也沒關係，get_key() 會在需要時當場詢問。
    """
    target = Path(path) if path else env_path()
    if not target.exists():
        return {}

    loaded = {}
    for raw in target.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if override or not os.environ.get(key):
            os.environ[key] = value
        loaded[key] = value
    return loaded


def _ask_key(name: str) -> str:
    """.env 沒有金鑰時，當場請使用者輸入（輸入內容不會留在 notebook 輸出裡）。"""
    try:
        import getpass

        value = getpass.getpass(f"請貼上 {name}（輸入不會顯示）：").strip()
    except Exception:
        return ""
    if value:
        os.environ[name] = value
    return value


# getpass 的提示只有「坐在 notebook 前面的人」看得到。
# 網頁服務跳這個提示，等於讓那個 HTTP 請求一直卡著等一個
# 沒有人看得到的輸入框 —— 所以服務啟動時會把它關掉。
_INTERACTIVE = True


def set_interactive(on: bool) -> None:
    """開關「找不到金鑰時當場詢問」。server/cinema/apps.py 會關掉它。"""
    global _INTERACTIVE
    _INTERACTIVE = bool(on)


def get_key(name: str, required: bool = True) -> str:
    """取得金鑰。順序：環境變數 -> .env -> 當場輸入。"""
    load_env()
    value = os.environ.get(name, "").strip()
    if value:
        return value
    if _INTERACTIVE:
        value = _ask_key(name)
    if not value and required:
        raise RuntimeError(
            f"缺少 {name}。兩種解法擇一：\n"
            f"  1. 在專案根目錄的 .env 填入 {name}=...\n"
            f"  2. 重跑這一格，在提示視窗貼上金鑰"
        )
    return value


def tmdb_token(required: bool = True) -> str:
    return get_key(TMDB_KEY_NAME, required=required)


def gemini_key(required: bool = True) -> str:
    return get_key(GEMINI_KEY_NAME, required=required)


# --------------------------------------------------------------------------
# 讓使用者換成自己的金鑰
# --------------------------------------------------------------------------
def has_key(name: str) -> bool:
    """這把金鑰現在有沒有值？不會跳出任何提示，純粹查詢。"""
    load_env()
    return bool(os.environ.get(name, "").strip())


def key_status() -> dict:
    """哪幾把金鑰已經設定好了。

    **只回布林值。** 金鑰本身絕不往外送 —— 網頁前端只需要知道
    「有沒有設定」，不需要也不應該拿到值。
    """
    return {"tmdb": has_key(TMDB_KEY_NAME), "gemini": has_key(GEMINI_KEY_NAME)}


_ENV_HEADER = [
    "# 這個檔案放金鑰，內容不會顯示在 notebook 的輸出裡。",
    "#",
    "# 可以直接編輯，也可以在網頁右上角的「API 金鑰」按鈕裡填 ——",
    "# 兩條路寫的是同一個檔案。",
    "",
]


def save_keys(**values) -> Path:
    """把金鑰寫回 .env，並立刻在目前這個行程生效。

    只換掉指定的那幾行，其他內容原封不動 —— 直接整份蓋掉的話，
    檔案裡的說明註解會在學員存第一次金鑰時就消失。

    值給空字串代表清掉那一把。回傳寫入的檔案路徑。
    """
    target = env_path()
    if target.exists():
        lines = target.read_text(encoding="utf-8").splitlines()
    else:
        # 檔案不存在時順手補上抬頭，不然學員存完金鑰打開來會是光禿禿兩行
        lines = list(_ENV_HEADER)

    for name, value in values.items():
        value = str(value or "").strip()
        for index, raw in enumerate(lines):
            line = raw.strip()
            if line.startswith("#") or "=" not in line:
                continue
            if line.partition("=")[0].strip() == name:
                lines[index] = f"{name}={value}"
                break
        else:
            lines.append(f"{name}={value}")
        # 同時更新環境變數，這樣不必重啟服務就生效
        os.environ[name] = value

    target.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return target


# --------------------------------------------------------------------------
# 每本 notebook 的開場
# --------------------------------------------------------------------------
def setup(requires=()) -> None:
    """每本 notebook 第一格呼叫：確認 sys.path、載入 .env、檢查前置模組。

    requires 列出這本 notebook 需要、但由前面 notebook 產生的模組。
    例如 03 需要 setup(requires=["sources", "tmdb"])，
    學員若跳著執行會得到明確的指示，而不是看不懂的 ImportError。
    """
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    load_env()

    missing = [m for m in requires if not (PKG_DIR / f"{m}.py").exists()]
    if missing:
        lines = [
            f"  - movieapp/{m}.py  ->  請先完整執行 {_MODULE_OWNER.get(m, '前面的 notebook')}"
            for m in missing
        ]
        raise RuntimeError("缺少前置模組，這本 notebook 還不能跑：\n" + "\n".join(lines))

    print(f"環境就緒｜根目錄：{ROOT}")


# --------------------------------------------------------------------------
# 表格排版小工具
# --------------------------------------------------------------------------
def display_width(text) -> int:
    """字串在終端機／notebook 裡佔幾格。

    中文字是全形，一個字佔兩格，但 len() 只算一個字元。
    直接用 ljust() 排版中英混雜的表格會歪掉，所以要另外算。
    """
    return sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in str(text))


def pad(text, width: int, align: str = "left") -> str:
    """把字串補到指定的顯示寬度（中文字算兩格）。"""
    text = str(text)
    space = " " * max(0, width - display_width(text))
    return space + text if align == "right" else text + space


# --------------------------------------------------------------------------
# 環境自我檢查
# --------------------------------------------------------------------------
def _has_module(name: str) -> bool:
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def doctor() -> bool:
    """印出環境檢查清單。全部通過回傳 True。

    課堂上先跑這一格，就知道自己的環境有沒有問題，
    不用等到第三本 notebook 才發現套件沒裝。
    """
    rows = []

    ok_py = sys.version_info >= (3, 9)
    rows.append((ok_py, "Python 版本", f"{sys.version.split()[0]}（需要 3.9 以上）"))

    for pkg, label in [
        ("requests", "requests 套件"),
        ("django", "Django 套件"),
        ("corsheaders", "django-cors-headers"),
        ("pandas", "pandas 套件"),
    ]:
        rows.append((_has_module(pkg), label,
                     "已安裝" if _has_module(pkg) else "缺少，請執行 %pip install -r ../requirements.txt"))

    for mod in ("config", "http"):
        exists = (PKG_DIR / f"{mod}.py").exists()
        rows.append((exists, f"movieapp/{mod}.py",
                     "已產生" if exists else "尚未產生（由 00_環境設定.ipynb 寫出）"))

    for mod, owner in _MODULE_OWNER.items():
        exists = (PKG_DIR / f"{mod}.py").exists()
        rows.append((exists, f"movieapp/{mod}.py", "已產生" if exists else f"尚未產生（由 {owner} 寫出）"))

    load_env()
    for label, key in [("TMDB 金鑰", TMDB_KEY_NAME), ("Gemini 金鑰", GEMINI_KEY_NAME)]:
        val = os.environ.get(key, "").strip()
        rows.append((bool(val), label, f"已設定（{val[:6]}…）" if val else "未設定，需要時會提示輸入"))

    page = ROOT / "server" / "static" / "index.html"
    rows.append((page.exists(), "前端 index.html",
                 "已產生" if page.exists() else "尚未產生（由 05_接成服務.ipynb 寫出）"))

    width = max(display_width(label) for _, label, _ in rows)
    print("環境檢查")
    print("=" * (width + 40))
    for ok, label, detail in rows:
        print(f"  {'[OK]' if ok else '[--]'}  {pad(label, width)}  {detail}")
    print("=" * (width + 40))

    hard_fail = not ok_py or not _has_module("requests")
    print("結論：" + ("環境沒問題，可以開始。" if not hard_fail
                    else "有必要項目未通過，請先處理上面標示 [--] 的項目。"))
    return not hard_fail

### `http.py`：唯一的對外出入口

注意 `fetch_json()` 的回傳值是 **`(data, error)`**，不是丟例外。

因為外部 API 失敗是「正常會發生的事」，不是程式寫錯。
把錯誤當成一般的值傳回去，呼叫端就不必到處包 `try/except`，
最後接成網頁服務時，也不會因為一家影城掛掉就整個噴 500。

這個回傳形式會貫穿整套教材，後面每一個函式都長這樣。

In [ ]:
%%writefile ../movieapp/http.py
"""所有對外的 HTTP 請求都必須走這裡。

為什麼要統一出入口？因為「錯誤處理」只要寫一次。
外部 API 會逾時、會斷線、會回 429、會回一坨不是 JSON 的東西，
這些狀況在這裡一次翻譯成看得懂的中文訊息，
呼叫端只要處理 (data, error) 這組回傳值就好。

之後想加快取、加重試、加日誌，也都只有這一個地方要改。

本檔案由 notebooks/00_環境設定.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

from __future__ import annotations

# 外部 API 的預設 timeout（秒）
DEFAULT_TIMEOUT = 20

# 影城的 API 沒有公開文件，不帶瀏覽器特徵的請求會被擋，
# 所以統一準備一組看起來像瀏覽器的 header。
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
    ),
    "Accept": "application/json",
}


def fetch_json(
    url: str,
    *,
    method: str = "GET",
    params: dict | None = None,
    headers: dict | None = None,
    json_body: dict | None = None,
    timeout: int | None = None,
):
    """抓取外部 API 並解析 JSON。

    回傳 (data, error)：成功時 error 為 None，失敗時 data 為 None。

    為什麼不丟例外？因為外部 API 失敗是「正常會發生的事」，不是程式寫錯。
    把錯誤當成一般的值傳回去，呼叫端就不必到處包 try/except，
    也不會因為一家影城掛掉就讓整個服務噴 500。
    """
    try:
        import requests
    except ImportError:
        return None, "沒有安裝 requests，請執行 %pip install -r ../requirements.txt"

    try:
        resp = requests.request(
            method,
            url,
            params=params,
            headers=headers,
            json=json_body,
            timeout=timeout or DEFAULT_TIMEOUT,
        )
        resp.raise_for_status()
        return resp.json(), None
    except requests.exceptions.Timeout:
        return None, f"外部 API 逾時（超過 {timeout or DEFAULT_TIMEOUT} 秒）"
    except requests.exceptions.ConnectionError:
        return None, "無法連線到外部 API，請檢查網路"
    except requests.exceptions.HTTPError as exc:
        code = exc.response.status_code
        if code == 401:
            return None, "外部 API 回應 401：金鑰無效或未帶上金鑰"
        if code == 429:
            return None, "外部 API 回應 429：請求太頻繁，配額或流量限制已達上限"
        return None, f"外部 API 回應錯誤：HTTP {code}"
    except ValueError:
        # requests 的 .json() 解析失敗時丟的是 ValueError（JSONDecodeError 是它的子類）
        return None, "外部 API 的回應不是有效的 JSON"
    except requests.exceptions.RequestException as exc:
        return None, f"請求失敗：{exc}"

---
## 3 ・ 順手把三個設定檔也寫出來

`requirements.txt` 讓你換一台電腦時能重建一模一樣的環境，
`.env.example` 是給要用自己金鑰的人看的範本，
`.env` 則是實際被讀取的那一份。

In [ ]:
%%writefile ../requirements.txt
# 課程用套件。第一次安裝在命令列執行（連同 Jupyter 一起）：
#   python -m pip install jupyter -r requirements.txt
#
# 已經在 notebook 裡的話用 %pip，不要用 !pip ——
# %pip 會裝到目前 kernel 的環境，!pip 是丟給系統 shell，
# 用 Anaconda 時兩者會裝到不同地方，然後 import 失敗且看不出原因。
#
# 本檔案由 notebooks/00_環境設定.ipynb 的 %%writefile 產生。

requests>=2.31
pandas>=2.0

# 05_接成服務.ipynb 才會用到
Django>=4.2,<5.1
django-cors-headers>=4.3

In [ ]:
%%writefile ../.env.example
# 複製成 .env 後填入自己的金鑰。
#
# 兩把都不是必要的：.env 不存在時，程式會在需要用到的那一刻
# 跳出輸入提示，貼上去就能繼續，輸入內容不會留在 notebook 的輸出裡。
#
# TMDB   : https://www.themoviedb.org/settings/api  取得 Read Access Token
# Gemini : https://aistudio.google.com/apikey       取得 API Key

TMDB_READ_ACCESS_TOKEN=
GEMINI_API_KEY=

`.env` 是真正被讀取的那一份，裡面放課程共用的金鑰。

**這一格會蓋掉現有的 `.env`。** 如果你已經填了自己的金鑰，
就跳過這一格，或先把下面的值換成你自己的再執行。

> 為什麼連 `.env` 都要用 `%%writefile` 寫出來？
> 因為它是隱藏檔，用雲端硬碟同步、解壓縮、或只複製 `notebooks/`
> 過去時很容易掉。掉了之後系統會在查 TMDB 的那一刻
> 跳出「請貼上 TMDB_READ_ACCESS_TOKEN」，很難聯想到原因。
> 寫在 notebook 裡，它就跟其他檔案一樣是「跑過就會有」。


In [ ]:
%%writefile ../.env
# 課程共用金鑰。兩把都是免費額度，隨教材一起發給學員。
#
# 若要改成讓學員各自申請，把下面兩行的值清空即可 ——
# config.get_key() 會在需要時跳出輸入提示，不必改任何程式碼。

TMDB_READ_ACCESS_TOKEN=***REMOVED***
GEMINI_API_KEY=***REMOVED***


---
## 4 ・ 開場四行

地基蓋好了，現在可以 import 它。**每一本 notebook 的第一格都是這四行**：
讓 Python 找得到 `movieapp` 套件、載入金鑰、確認前面章節的產出都在。

`%autoreload 2` 讓你改完 `.py` 檔之後，不用重開 kernel 就生效。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")
from movieapp.config import setup; setup()

---
## 5 ・ 環境檢查

`doctor()` 會列出每一項的狀態。現在跑會看到 `sources.py`～`gemini.py`
和前端 `index.html` 都標示「尚未產生」—— 這是正常的，
它們要等你做完對應的章節才會出現。

必須通過的只有兩項：**Python 版本**和 **requests 套件**。

In [ ]:
from movieapp import config
config.doctor();

---
## 6 ・ 金鑰

這套教材會用到兩個外部服務的金鑰：

| 服務 | 用途 | 額度 |
|---|---|---|
| TMDB | 電影評分、海報、類型 | 免費，唯讀 |
| Gemini | AI 聊天回答 | 免費層，**額度綁在金鑰上** |

金鑰放在專案根目錄的 `.env`，課程已經幫你填好了。讀取順序是：

```
環境變數  ->  .env  ->  當場輸入（不會留在 notebook 裡）
```

想用自己的金鑰，把 `.env` 的值清空，執行時就會跳出輸入框。
申請網址寫在剛剛產生的 `.env.example` 裡。

> **關於 Gemini 的額度**：免費層是**按金鑰計算**的，全班共用一把時，
> 大家同時發問很容易一起撞到上限，收到 429 錯誤。
> 遇到的話等幾分鐘，或換成自己的金鑰。

In [ ]:
# 確認金鑰讀得到（只顯示前幾碼，不會把完整金鑰印出來）
for name in [config.TMDB_KEY_NAME, config.GEMINI_KEY_NAME]:
    value = config.get_key(name, required=False)
    print(f"  {name:26} {'已設定 ' + value[:8] + '…' if value else '未設定'}")

---
## 7 ・ 連線測試

最後確認四支外部 API 都通。

**這一步不能跳過。** 這套教材全程即時連線，沒有離線備援 ——
如果這裡就連不上，後面的章節會一路失敗。

注意這一格已經在用 `http.fetch_json()` 了：四支 API 三種協定、
兩種方法、各自的 header 需求，呼叫端寫起來卻長得一模一樣。
這就是統一出入口的好處。

In [ ]:
from movieapp import http
from movieapp.config import pad

targets = [
    ("秀泰影城",  "https://capi.showtimes.com.tw/4/app/bootstrap", "GET"),
    ("美麗華影城", "https://www.miramarcinemas.tw/api/Booking/GetMovie/", "POST"),
    ("TMDB",     "https://api.themoviedb.org/3/genre/movie/list", "GET"),
    ("Gemini",   "https://generativelanguage.googleapis.com/v1beta/interactions", "POST"),
]

for label, url, method in targets:
    data, error = http.fetch_json(
        url, method=method,
        headers=dict(http.BROWSER_HEADERS,
                     Authorization=f"Bearer {config.tmdb_token(required=False)}",
                     Referer="https://www.miramarcinemas.tw/"),
        json_body={} if method == "POST" else None, timeout=15)
    # 回 400／401 代表對方有回應，只是這一次的請求內容不完整 ——
    # 對「連得上嗎」這個問題來說，那也算通。
    reachable = data is not None or any(c in str(error) for c in ("400", "401"))
    print(f"  {pad(label, 12)} {'連得上' if reachable else '連不上：' + str(error)}")

---
## 準備好了

上面四支都「連得上」就可以開始了。

**下一步：打開 `01_影城API.ipynb`。**

課堂上遇到問題時，先回來跑一次 `config.doctor()`，
八成的狀況它都能告訴你原因。